# 5.14 Uygulama: Yüz Algılama Hattı

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/05-sklearn/14-image-features.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 05.14 Image Features

Kitabın bu bölümü makine öğrenmesinin merkezi kavram ve algoritmalarının birçoğunu inceledi. Ancak bu kavramlardan gerçek dünya uygulamasına geçmek zor olabilir. Gerçek veri kümeleri gürültülü ve heterojendir; eksik öznitelikler olabilir, veri temiz [n_samples, n_features] matrisine dönüştürmek zor olabilir. Burada tartıştığımız yöntemleri uygulamadan önce veriden öznitelik çıkarmanız gerekir — tüm alanlara uyan tek bir formül yoktur; veri bilimci olarak kendi sezginiz ve uzmanlığınızı kullanırsınız.

Makine öğrenmesinin ilginç uygulamalarından biri görüntülerdir; piksel düzeyinde özniteliklerle sınıflandırma örneklerini görmüştük. Gerçek dünya verisi nadiren bu kadar düzenlidir; basit pikseller yeterli olmaz — bu da görüntü verisi için geniş bir öznitelik çıkarma literatürüne yol açmıştır (bkz. 5.4 Öznitelik Mühendisliği).

Bu bölümde bir öznitelik çıkarma tekniğine bakacağız: yönelim gradyan histogramı (HOG). HOG, aydınlatma gibi karıştırıcı faktörlerden bağımsız geniş bilgi taşıyan görüntü özelliklerine duyarlı vektör temsiline dönüştürür. Bu özniteliklerle kitabın bu bölümünde gördüğümüz makine öğrenmesi algoritmaları ve kavramlarıyla basit bir yüz algılama hattı geliştireceğiz.

Standart içe aktarmalarla başlayalım:


In [ ]:
# imports_hog.py
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
import numpy as np



> **Not**
>

## HOG Öznitelikleri

HOG, görüntülerde yaya tanıma bağlamında geliştirilmiş doğrudan bir öznitelik çıkarma prosedürüdür. Adımlar:

Hızlı bir HOG çıkarıcı Scikit-Image projesinde yerleşiktir; her hücredeki yönelim gradyanlarını görece hızlıca deneyip görselleştirebiliriz (aşağıdaki şekil):


In [ ]:
# hog_chelsea.py
from skimage import data, color, feature
import skimage.data

image = color.rgb2gray(data.chelsea())
hog_vec, hog_vis = feature.hog(image, visualize=True)

fig, ax = plt.subplots(1, 2, figsize=(12, 6),
                       subplot_kw=dict(xticks=[], yticks=[]))
ax[0].imshow(image, cmap='gray')
ax[0].set_title('input image')

ax[1].imshow(hog_vis)
ax[1].set_title('visualization of HOG features');



### 🧪 Şimdi deneyin

🧪 
      Basit HOG vektör boyutunu kontrol edin (scikit-image gerekir):
          
      # Pyodide'da skimage yoksa yerel notebook kullanın
try:
    from skimage import data, color, feature
    image = color.rgb2gray(data.chelsea())
    hog_vec, _ = feature.hog(image, pixels_per_cell=(16, 16), visualize=True)
    print("HOG boyutu:", hog_vec.shape)
except ImportError:
    print("skimage bu ortamda yok — yerel Jupyter önerilir")

## HOG Uygulamada: Basit Yüz Dedektörü

Bu HOG öznitelikleriyle herhangi bir Scikit-Learn tahmincisiyle basit yüz algılama algoritması kurulabilir; burada doğrusal destek vektör makinesi kullanacağız (5.7 SVM hatırlatması). Adımlar:

Adımları uygulayıp deneyelim.

### 1. Pozitif Eğitim Örnekleri

Çeşitli yüzleri gösteren pozitif eğitim örnekleriyle başlayacağız. Kolay bir veri kümesi Labeled Faces in the Wild; Scikit-Learn ile indirilebilir:


In [ ]:
# fetch_lfw_people.py
from sklearn.datasets import fetch_lfw_people
faces = fetch_lfw_people()
positive_patches = faces.images
positive_patches.shape



Bu, eğitim için 13.000 yüz görüntüsü verir.

### 2. Negatif Eğitim Örnekleri

Sonra yüz içermeyen, benzer boyutlu küçük resimlere ihtiyacımız var. Girdi görüntü küpusundan çeşitli ölçeklerde küçük resimler çıkarılabilir. Burada Scikit-Image ile gelen görüntüleri ve Scikit-Learn PatchExtractor'ı kullanacağız:


In [ ]:
# camera_shape.py
data.camera().shape



In [ ]:
# negative_patches.py
from skimage import data, transform

imgs_to_use = ['camera', 'text', 'coins', 'moon',
               'page', 'clock', 'immunohistochemistry',
               'chelsea', 'coffee', 'hubble_deep_field']
raw_images = (getattr(data, name)() for name in imgs_to_use)
images = [color.rgb2gray(image) if image.ndim == 3 else image
          for image in raw_images]



In [ ]:
# extract_patches_fn.py
from sklearn.feature_extraction.image import PatchExtractor

def extract_patches(img, N, scale=1.0, patch_size=positive_patches[0].shape):
    extracted_patch_size = tuple((scale * np.array(patch_size)).astype(int))
    extractor = PatchExtractor(patch_size=extracted_patch_size,
                               max_patches=N, random_state=0)
    patches = extractor.transform(img[np.newaxis])
    if scale != 1:
        patches = np.array([transform.resize(patch, patch_size)
                            for patch in patches])
    return patches

negative_patches = np.vstack([extract_patches(im, 1000, scale)
                              for im in images for scale in [0.5, 1.0, 2.0]])
negative_patches.shape



Artık yüz içermeyen 30.000 uygun görüntü yamasına sahibiz. Birkaçını görselleştirelim (aşağıdaki şekil):


In [ ]:
# negative_patches_plot.py
fig, ax = plt.subplots(6, 10)
for i, axi in enumerate(ax.flat):
    axi.imshow(negative_patches[500 * i], cmap='gray')
    axi.axis('off')



Umarız bunlar algoritmanın göreceği "yüz olmayan" uzayını yeterince kapsar.

### 3. Kümeleri Birleştirip HOG Çıkarma

Pozitif ve negatif örnekler hazır; birleştirip HOG özniteliklerini hesaplayabiliriz. Her görüntü için önemli hesaplama gerektiğinden bu adım biraz sürer:


In [ ]:
# hog_features_train.py
from itertools import chain
X_train = np.array([feature.hog(im)
                    for im in chain(positive_patches,
                                    negative_patches)])
y_train = np.zeros(X_train.shape[0])
y_train[:positive_patches.shape[0]] = 1



In [ ]:
# X_train_shape.py
X_train.shape



1.215 boyutta 43.000 eğitim örneğimiz kaldı; veriyi Scikit-Learn'e besleyebilecek biçimdeyiz!

> **Not**
>

### 4. Destek Vektör Makinesi Eğitimi

Burada öğrendiğimiz araçlarla küçük resim yamalarını sınıflandıran bir model oluşturacağız. Bu yüksek boyutlu ikili sınıflandırma için doğrusal SVM iyi seçimdir. Büyük örnek sayısında SVC'ye kıyasla genelde daha iyi ölçeklenen LinearSVC kullanacağız.

Önce hızlı bir taban çizgi için basit Gauss naive Bayes tahmincisi deneyelim:


In [ ]:
# gnb_baseline_faces.py
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score

cross_val_score(GaussianNB(), X_train, y_train)



Eğitim verisinde basit naive Bayes bile %95'in üzerinde doğruluk veriyor. Destek vektör makinesini deneyelim; C parametresi üzerinde grid search yapalım:


In [ ]:
# linearsvc_grid_faces.py
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(LinearSVC(), {'C': [1.0, 2.0, 4.0, 8.0]})
grid.fit(X_train, y_train)
grid.best_score_



In [ ]:
# linearsvc_best_c.py
grid.best_params_



Bu bizi neredeyse %99 doğruluğa taşır. En iyi tahminciyi alıp tüm veri kümesinde yeniden eğitelim:


In [ ]:
# linearsvc_final_fit.py
model = grid.best_estimator_
model.fit(X_train, y_train)



### 5. Yeni Görüntüde Yüz Bulma

Model hazır; yeni bir görüntü alıp modelin nasıl yaptığına bakalım. Basitlik için aşağıdaki şekildeki astronot görüntüsünün bir bölümünü kullanacağız; kayan pencere çalıştırıp her yamayı değerlendireceğiz:


In [ ]:
# test_astronaut_crop.py
test_image = skimage.data.astronaut()
test_image = skimage.color.rgb2gray(test_image)
test_image = skimage.transform.rescale(test_image, 0.5)
test_image = test_image[:160, 40:180]

plt.imshow(test_image, cmap='gray')
plt.axis('off');



Sonra görüntü yamaları üzerinde yineleyen bir pencere oluşturup her yama için HOG özniteliklerini hesaplayalım:


In [ ]:
# sliding_window_fn.py
def sliding_window(img, patch_size=positive_patches[0].shape,
                   istep=2, jstep=2, scale=1.0):
    Ni, Nj = (int(scale * s) for s in patch_size)
    for i in range(0, img.shape[0] - Ni, istep):
        for j in range(0, img.shape[1] - Ni, jstep):
            patch = img[i:i + Ni, j:j + Nj]
            if scale != 1:
                patch = transform.resize(patch, patch_size)
            yield (i, j), patch
            
indices, patches = zip(*sliding_window(test_image))
patches_hog = np.array([feature.hog(patch) for patch in patches])
patches_hog.shape



Son olarak HOG'lu yamaları modelle değerlendirip hangisinin yüz içerdiğini görelim:


In [ ]:
# predict_patches.py
labels = model.predict(patches_hog)
labels.sum()



Yaklaşık 2.000 yamadan 48 algılama bulduk. Yamalar hakkındaki bilgiyle test görüntüsünde dikdörtgen olarak gösterelim (aşağıdaki şekil):


In [ ]:
# draw_detections.py
fig, ax = plt.subplots()
ax.imshow(test_image, cmap='gray')
ax.axis('off')

Ni, Nj = positive_patches[0].shape
indices = np.array(indices)

for i, j in indices[labels == 1]:
    ax.add_patch(plt.Rectangle((j, i), Nj, Ni, edgecolor='red',
                               alpha=0.3, lw=2, facecolor='none'))



Algılanan yamaların hepsi örtüşüyor ve görüntüdeki yüzü buldu! Birkaç satır Python için fena değil.

## Uyarılar ve İyileştirmeler

Önceki kod ve örneklere biraz daha inerseniz, üretime hazır yüz dedektörü iddiasında olmadan önce hâlâ işimiz olduğunu görürsünüz. Birkaç sorun ve iyileştirme:

Eğitim kümemiz, özellikle negatif öznitelikler için, pek eksiksiz değil

Temel sorun, eğitim kümesinde olmayan birçok yüz benzeri dokunun olması; mevcut model yanlış pozitiflere çok yatkın. Tam astronot görüntüsünde denerseniz model başka bölgelerde birçok yanlış algılama üretir. Negatif eğitim kümesine daha geniş görüntü seti eklemek iyileşme sağlayabilir. Hard negative mining de seçenektir: sınıflandırıcının görmediği yeni görüntülerde yanlış pozitif yamaları bulup negatif örnek olarak eğitime ekleyip yeniden eğitmek.

Mevcut hat yalnızca tek ölçekte arıyor

Şu anki yazım yaklaşık 62×47 piksel olmayan yüzleri kaçırır. Çeşitli boyutlarda kayan pencere ve modele vermeden önce skimage.transform.resize ile yeniden boyutlandırma ile ele alınabilir; buradaki sliding_window yardımcısı bunun için tasarlanmıştır.

Örtüşen algılama yamalarını birleştirmeliyiz

Üretim hattında aynı yüzün 30 algılamasını değil, örtüşen grupları tek algılamaya indirmek isteriz — mean shift kümeleme veya makine görüşünde yaygın non-maximum suppression.

Hat akıcı hale getirilmeli

Önceki sorunlar giderildikten sonra eğitim görüntülerini alıp kayan pencere çıktısı üreten daha akıcı bir pipeline kurmak istenebilir — Python veri biliminde güçlü yön budur.

Daha yeni gelişmeler: derin öğrenme

Son olarak, makine öğrenmesinde HOG ve benzeri prosedürel öznitelik çıkarma her zaman kullanılmaz; birçok modern nesne algılama hattı derin sinir ağı varyantlarını (derin öğrenme) kullanır: sinir ağları, kullanıcı sezgisine değil veriden optimal öznitelik stratejilerini öğrenen tahminciler olarak düşünülebilir.

Alan son yıllarda harika sonuçlar verse de derin öğrenme önceki bölümlerdeki modellerden kavramsal olarak çok farklı değildir; asıl ilerleme, çok daha büyük eğitim veri kütleleri üzerinde çok daha esnek modeller eğitmek için modern donanım kullanmaktır. Ölçek farklı olsa da amaç aynıdır: veriden model kurmak.

Daha ileri gitmek istiyorsanız 5.15 Kaynaklar bölümündeki referans listesi iyi bir başlangıç noktasıdır!

> **Not**
>
